# Prompt Caching Cost Optimization

**Week 2 Day 1 - Learning Lab**

Understanding and implementing prompt caching to reduce API costs.

## Intent

Learn to:
- Understand how prompt caching works
- Implement caching for different providers (OpenAI, Anthropic, Gemini)
- Measure actual cost savings
- Structure prompts for optimal caching

## Expected Insights

- Cost savings from caching (can be 70%+ on repeated calls)
- Provider-specific caching implementations
- Best practices for prompt structure


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from litellm import completion

load_dotenv(override=True)

# Setup clients
openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_client = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

print("Clients initialized")


## Experiment: Caching Large Static Context

We'll use a large text (Hamlet) as static context and see the cost difference between first and second calls.


In [ ]:
# Load a large text file (or use a sample)
# In the course, we used hamlet.txt
# For this experiment, we'll create a sample large context

large_context = """
This is a large static context that will be cached. 
It contains instructions, examples, and reference material.
""" * 1000  # Simulate large context

question = "What is the main topic of this context?"

# First call - no caching
messages_first = [
    {"role": "user", "content": question},
    {"role": "user", "content": f"For context, here is the full text:\n\n{large_context}"}
]

print("Making first call (no cache)...")
response_first = completion(
    model="gemini/gemini-2.5-flash-lite",
    messages=messages_first
)

print(f"First call:")
print(f"  Input tokens: {response_first.usage.prompt_tokens}")
print(f"  Output tokens: {response_first.usage.completion_tokens}")
print(f"  Total tokens: {response_first.usage.total_tokens}")
if hasattr(response_first, '_hidden_params') and 'response_cost' in response_first._hidden_params:
    cost_first = response_first._hidden_params['response_cost'] * 100
    print(f"  Cost: {cost_first:.4f} cents")


In [ ]:
# Second call - should use cache
print("\nMaking second call (should use cache)...")
response_second = completion(
    model="gemini/gemini-2.5-flash-lite",
    messages=messages_first  # Same messages
)

print(f"Second call:")
print(f"  Input tokens: {response_second.usage.prompt_tokens}")
print(f"  Output tokens: {response_second.usage.completion_tokens}")
if hasattr(response_second.usage, 'prompt_tokens_details') and hasattr(response_second.usage.prompt_tokens_details, 'cached_tokens'):
    cached = response_second.usage.prompt_tokens_details.cached_tokens
    print(f"  Cached tokens: {cached}")
if hasattr(response_second, '_hidden_params') and 'response_cost' in response_second._hidden_params:
    cost_second = response_second._hidden_params['response_cost'] * 100
    print(f"  Cost: {cost_second:.4f} cents")
    
    if 'cost_first' in locals():
        savings = ((cost_first - cost_second) / cost_first) * 100
        print(f"\n  💰 Savings: {savings:.1f}%")


## Key Takeaways

1. **Prompt caching requires exact prefix matches** - Static content must be identical
2. **Structure matters** - Place static content at beginning, variable at end
3. **Provider differences:**
   - OpenAI: 4x cheaper for cached content
   - Anthropic: 25% more to prime, 10x cheaper to reuse
   - Gemini: Automatic caching for repeated prefixes
4. **Real savings:** Can save 70%+ on repeated calls with large static context
